# Black Grouse 2030 Land-Use Scenario Project

## Stage 7: Core-Habitat Connectivity Analysis

This notebook compares the spatial connectivity of:

- the 2024 baseline;
- dispersed restoration;
- patch-enlargement restoration;
- connectivity-focused restoration;
- integrated low-matrix-pressure restoration.

All four 2030 scenarios contain the same restored habitat area. Differences in connectivity therefore reflect the spatial configuration of restoration rather than differences in habitat amount.

Connectivity will be evaluated using habitat-network components at multiple maximum gap distances.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import rasterio

from scipy import ndimage

print("Packages imported successfully.")

Packages imported successfully.


In [2]:
PROJECT_DIR = Path.cwd()

PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
TABLES_DIR = PROJECT_DIR / "outputs" / "tables"

CORE_HABITAT_CLASS = 2

NETWORK_RASTERS = {
    "baseline_2024": (
        PROCESSED_DIR / "LGN2024_harmonised_25m.tif"
    ),
    "dispersed": (
        PROCESSED_DIR / "landuse_2030_dispersed.tif"
    ),
    "patch_enlargement": (
        PROCESSED_DIR
        / "landuse_2030_patch_enlargement.tif"
    ),
    "connectivity_focused": (
        PROCESSED_DIR
        / "landuse_2030_connectivity_focused.tif"
    ),
    "integrated_low_matrix_pressure": (
        PROCESSED_DIR
        / "landuse_2030_integrated_low_matrix_pressure.tif"
    ),
}

for scenario_name, raster_path in NETWORK_RASTERS.items():

    status = "FOUND" if raster_path.exists() else "MISSING"

    print(f"{scenario_name}: {status}")
    print(raster_path)

baseline_2024: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\LGN2024_harmonised_25m.tif
dispersed: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_dispersed.tif
patch_enlargement: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_patch_enlargement.tif
connectivity_focused: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_connectivity_focused.tif
integrated_low_matrix_pressure: FOUND
C:\Users\smit1\BlackGrouse_2030\data\processed\landuse_2030_integrated_low_matrix_pressure.tif


In [3]:
network_arrays = {}
raster_profiles = {}

reference_shape = None
reference_transform = None
reference_crs = None
reference_mask = None

for scenario_name, raster_path in NETWORK_RASTERS.items():

    with rasterio.open(raster_path) as src:

        raster_data = src.read(1)

        network_arrays[scenario_name] = raster_data
        raster_profiles[scenario_name] = src.profile.copy()

        current_mask = raster_data != 0

        if reference_shape is None:

            reference_shape = raster_data.shape
            reference_transform = src.transform
            reference_crs = src.crs
            reference_mask = current_mask.copy()

        else:

            if raster_data.shape != reference_shape:
                raise ValueError(
                    f"{scenario_name} has a different shape."
                )

            if src.transform != reference_transform:
                raise ValueError(
                    f"{scenario_name} has a different transform."
                )

            if src.crs != reference_crs:
                raise ValueError(
                    f"{scenario_name} has a different CRS."
                )

            if not np.array_equal(
                current_mask,
                reference_mask,
            ):
                raise ValueError(
                    f"{scenario_name} has a different analysis mask."
                )

        core_pixels = int(
            np.sum(
                raster_data == CORE_HABITAT_CLASS
            )
        )

        print(f"\nScenario: {scenario_name}")
        print(f"Shape: {raster_data.shape}")
        print(f"CRS: {src.crs}")
        print(f"Resolution: {src.res}")
        print(f"Core-habitat pixels: {core_pixels:,}")


PIXEL_SIZE_METRES = abs(
    reference_transform.a
)

PIXEL_AREA_KM2 = (
    PIXEL_SIZE_METRES
    * PIXEL_SIZE_METRES
    / 1_000_000
)

print("\nAll connectivity-analysis rasters are aligned.")
print(f"Pixel size: {PIXEL_SIZE_METRES:.0f} m")


Scenario: baseline_2024
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Core-habitat pixels: 87,052

Scenario: dispersed
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Core-habitat pixels: 105,701

Scenario: patch_enlargement
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Core-habitat pixels: 105,701

Scenario: connectivity_focused
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Core-habitat pixels: 105,701

Scenario: integrated_low_matrix_pressure
Shape: (1592, 1023)
CRS: EPSG:28992
Resolution: (25.0, 25.0)
Core-habitat pixels: 105,701

All connectivity-analysis rasters are aligned.
Pixel size: 25 m


In [4]:
GAP_DISTANCES_METRES = [
    0,
    100,
    250,
    500,
    1000,
]


def create_circular_kernel(radius_pixels):
    """
    Create a circular binary structuring element.
    """

    rows, columns = np.ogrid[
        -radius_pixels:radius_pixels + 1,
        -radius_pixels:radius_pixels + 1,
    ]

    return (
        rows**2 + columns**2
        <= radius_pixels**2
    )


def calculate_network_metrics(
    raster_data,
    analysis_mask,
    scenario_name,
    gap_distance_metres,
):
    """
    Calculate habitat-network metrics for a specified
    approximate maximum gap distance.

    Existing habitat patches are defined using eight-neighbour
    connectivity. Habitat patches are assigned to the same
    network component when their approximate edge-to-edge gap
    is within the specified distance.
    """

    core_mask = (
        (raster_data == CORE_HABITAT_CLASS)
        & analysis_mask
    )

    connectivity_structure = np.ones(
        (3, 3),
        dtype=np.uint8,
    )

    patch_labels, original_patch_count = ndimage.label(
        core_mask,
        structure=connectivity_structure,
    )

    if gap_distance_metres == 0:

        expanded_core = core_mask.copy()

    else:

        expansion_radius_pixels = int(
            np.ceil(
                gap_distance_metres
                / (2 * PIXEL_SIZE_METRES)
            )
        )

        circular_kernel = create_circular_kernel(
            expansion_radius_pixels
        )

        expanded_core = ndimage.binary_dilation(
            core_mask,
            structure=circular_kernel,
            iterations=1,
            mask=analysis_mask,
        )

    component_labels, network_component_count = ndimage.label(
        expanded_core,
        structure=connectivity_structure,
    )

    component_habitat_pixels = np.bincount(
        component_labels[core_mask],
        minlength=network_component_count + 1,
    )[1:]

    component_habitat_pixels = component_habitat_pixels[
        component_habitat_pixels > 0
    ]

    component_areas_km2 = (
        component_habitat_pixels
        * PIXEL_AREA_KM2
    )

    total_habitat_area_km2 = (
        core_mask.sum()
        * PIXEL_AREA_KM2
    )

    largest_component_area_km2 = (
        component_areas_km2.max()
    )

    largest_component_habitat_percent = (
        largest_component_area_km2
        / total_habitat_area_km2
        * 100
    )

    mean_component_area_km2 = (
        component_areas_km2.mean()
    )

    patch_ids = np.arange(
        1,
        original_patch_count + 1,
    )

    patch_to_component = ndimage.maximum(
        component_labels,
        labels=patch_labels,
        index=patch_ids,
    ).astype(int)

    patches_per_component = np.bincount(
        patch_to_component,
        minlength=network_component_count + 1,
    )[1:]

    isolated_patch_count = int(
        np.sum(patches_per_component == 1)
    )

    isolated_patch_percentage = (
        isolated_patch_count
        / original_patch_count
        * 100
    )

    mean_patches_per_component = (
        original_patch_count
        / network_component_count
    )

    return {
        "scenario": scenario_name,
        "gap_distance_m": gap_distance_metres,
        "core_habitat_area_km2": round(
            total_habitat_area_km2,
            3,
        ),
        "original_patch_count": int(
            original_patch_count
        ),
        "network_component_count": int(
            network_component_count
        ),
        "isolated_patch_count": isolated_patch_count,
        "isolated_patch_percentage": round(
            isolated_patch_percentage,
            2,
        ),
        "mean_patches_per_component": round(
            mean_patches_per_component,
            3,
        ),
        "mean_component_area_km2": round(
            mean_component_area_km2,
            3,
        ),
        "largest_component_area_km2": round(
            largest_component_area_km2,
            3,
        ),
        "largest_component_habitat_percent": round(
            largest_component_habitat_percent,
            2,
        ),
    }

In [5]:
connectivity_records = []

for scenario_name, raster_data in network_arrays.items():

    print(f"\nCalculating connectivity: {scenario_name}")

    for gap_distance in GAP_DISTANCES_METRES:

        metrics = calculate_network_metrics(
            raster_data=raster_data,
            analysis_mask=reference_mask,
            scenario_name=scenario_name,
            gap_distance_metres=gap_distance,
        )

        connectivity_records.append(metrics)

        print(
            f"Gap {gap_distance:>4} m | "
            f"components: "
            f"{metrics['network_component_count']:,} | "
            f"largest component: "
            f"{metrics['largest_component_habitat_percent']:.2f}%"
        )


connectivity_metrics = pd.DataFrame(
    connectivity_records
)

CONNECTIVITY_OUTPUT_PATH = (
    TABLES_DIR
    / "core_habitat_connectivity_by_gap_distance.csv"
)

connectivity_metrics.to_csv(
    CONNECTIVITY_OUTPUT_PATH,
    index=False,
)

display(connectivity_metrics)

print("\nConnectivity table saved:")
print(CONNECTIVITY_OUTPUT_PATH)


Calculating connectivity: baseline_2024
Gap    0 m | components: 2,570 | largest component: 24.06%
Gap  100 m | components: 1,190 | largest component: 25.62%
Gap  250 m | components: 570 | largest component: 28.17%
Gap  500 m | components: 202 | largest component: 53.02%
Gap 1000 m | components: 17 | largest component: 98.29%

Calculating connectivity: dispersed
Gap    0 m | components: 19,058 | largest component: 19.83%
Gap  100 m | components: 6,526 | largest component: 21.21%
Gap  250 m | components: 67 | largest component: 99.42%
Gap  500 m | components: 4 | largest component: 99.99%
Gap 1000 m | components: 1 | largest component: 100.00%

Calculating connectivity: patch_enlargement
Gap    0 m | components: 2,290 | largest component: 20.31%
Gap  100 m | components: 1,098 | largest component: 21.75%
Gap  250 m | components: 508 | largest component: 36.48%
Gap  500 m | components: 169 | largest component: 50.61%
Gap 1000 m | components: 14 | largest component: 97.77%

Calculating co

,scenario,gap_distance_m,core_habitat_area_km2,original_patch_count,network_component_count,isolated_patch_count,isolated_patch_percentage,mean_patches_per_component,mean_component_area_km2,largest_component_area_km2,largest_component_habitat_percent
0,baseline_2024,0,54.408,2570,2570,2570,100.00,1.000,0.021,13.093,24.06
1,baseline_2024,100,54.408,2570,1190,816,31.75,2.160,0.046,13.941,25.62
2,baseline_2024,250,54.408,2570,570,325,12.65,4.509,0.095,15.329,28.17
3,baseline_2024,500,54.408,2570,202,88,3.42,12.723,0.269,28.848,53.02
4,baseline_2024,1000,54.408,2570,17,8,0.31,151.176,3.200,53.477,98.29
5,dispersed,0,66.063,19058,19058,19058,100.00,1.000,0.003,13.101,19.83
6,dispersed,100,66.063,19058,6526,3307,17.35,2.920,0.010,14.009,21.21
7,dispersed,250,66.063,19058,67,37,0.19,284.448,0.986,65.681,99.42
8,dispersed,500,66.063,19058,4,3,0.02,4764.500,16.516,66.059,99.99
9,dispersed,1000,66.063,19058,1,0,0.00,19058.000,66.063,66.063,100.00



Connectivity table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\core_habitat_connectivity_by_gap_distance.csv


In [6]:
baseline_connectivity = (
    connectivity_metrics[
        connectivity_metrics["scenario"] == "baseline_2024"
    ]
    .set_index("gap_distance_m")
)

comparison_records = []

SCENARIO_NAMES = [
    "dispersed",
    "patch_enlargement",
    "connectivity_focused",
    "integrated_low_matrix_pressure",
]

COMPARISON_METRICS = [
    "network_component_count",
    "isolated_patch_count",
    "isolated_patch_percentage",
    "mean_patches_per_component",
    "mean_component_area_km2",
    "largest_component_area_km2",
    "largest_component_habitat_percent",
]

for scenario_name in SCENARIO_NAMES:

    scenario_data = (
        connectivity_metrics[
            connectivity_metrics["scenario"] == scenario_name
        ]
        .set_index("gap_distance_m")
    )

    for gap_distance in GAP_DISTANCES_METRES:

        for metric_name in COMPARISON_METRICS:

            baseline_value = float(
                baseline_connectivity.loc[
                    gap_distance,
                    metric_name,
                ]
            )

            scenario_value = float(
                scenario_data.loc[
                    gap_distance,
                    metric_name,
                ]
            )

            absolute_change = (
                scenario_value - baseline_value
            )

            if baseline_value != 0:
                percentage_change = (
                    absolute_change
                    / baseline_value
                    * 100
                )
            else:
                percentage_change = np.nan

            comparison_records.append(
                {
                    "scenario": scenario_name,
                    "gap_distance_m": gap_distance,
                    "metric": metric_name,
                    "baseline_value": baseline_value,
                    "scenario_value": scenario_value,
                    "absolute_change": round(
                        absolute_change,
                        3,
                    ),
                    "percentage_change": round(
                        percentage_change,
                        2,
                    ),
                }
            )


connectivity_changes = pd.DataFrame(
    comparison_records
)

CONNECTIVITY_CHANGE_OUTPUT = (
    TABLES_DIR
    / "connectivity_changes_relative_to_2024.csv"
)

connectivity_changes.to_csv(
    CONNECTIVITY_CHANGE_OUTPUT,
    index=False,
)

display(connectivity_changes)

print("\nConnectivity-change table saved:")
print(CONNECTIVITY_CHANGE_OUTPUT)

,scenario,gap_distance_m,metric,baseline_value,scenario_value,absolute_change,percentage_change
0,dispersed,0,network_component_count,2570.000,19058.000,16488.000,641.56
1,dispersed,0,isolated_patch_count,2570.000,19058.000,16488.000,641.56
2,dispersed,0,isolated_patch_percentage,100.000,100.000,0.000,0.00
3,dispersed,0,mean_patches_per_component,1.000,1.000,0.000,0.00
4,dispersed,0,mean_component_area_km2,0.021,0.003,-0.018,-85.71
...,...,...,...,...,...,...,...
135,integrated_low_matrix_pressure,1000,isolated_patch_percentage,0.310,0.310,0.000,0.00
136,integrated_low_matrix_pressure,1000,mean_patches_per_component,151.176,151.412,0.236,0.16
137,integrated_low_matrix_pressure,1000,mean_component_area_km2,3.200,3.886,0.686,21.44
138,integrated_low_matrix_pressure,1000,largest_component_area_km2,53.477,65.128,11.651,21.79



Connectivity-change table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\connectivity_changes_relative_to_2024.csv


In [7]:
expected_scenarios = set(NETWORK_RASTERS.keys())
expected_gap_distances = set(GAP_DISTANCES_METRES)

validation_records = []

for scenario_name in NETWORK_RASTERS:

    scenario_results = connectivity_metrics[
        connectivity_metrics["scenario"] == scenario_name
    ]

    observed_gap_distances = set(
        scenario_results["gap_distance_m"].tolist()
    )

    habitat_areas = scenario_results[
        "core_habitat_area_km2"
    ].unique()

    validation_records.append(
        {
            "scenario": scenario_name,
            "number_of_gap_distances": len(
                observed_gap_distances
            ),
            "all_gap_distances_present": (
                observed_gap_distances
                == expected_gap_distances
            ),
            "single_habitat_area_value": (
                len(habitat_areas) == 1
            ),
            "core_habitat_area_km2": float(
                habitat_areas[0]
            ),
            "component_counts_positive": (
                scenario_results[
                    "network_component_count"
                ] > 0
            ).all(),
            "largest_component_percent_valid": (
                (
                    scenario_results[
                        "largest_component_habitat_percent"
                    ] >= 0
                )
                & (
                    scenario_results[
                        "largest_component_habitat_percent"
                    ] <= 100
                )
            ).all(),
        }
    )


connectivity_validation = pd.DataFrame(
    validation_records
)

scenario_habitat_areas = (
    connectivity_validation[
        connectivity_validation["scenario"]
        != "baseline_2024"
    ]["core_habitat_area_km2"]
)

equal_scenario_habitat_area = (
    scenario_habitat_areas.nunique() == 1
)


VALIDATION_OUTPUT_PATH = (
    TABLES_DIR
    / "connectivity_analysis_validation.csv"
)

connectivity_validation.to_csv(
    VALIDATION_OUTPUT_PATH,
    index=False,
)

display(connectivity_validation)

print(
    "\nEqual habitat area across all 2030 scenarios:",
    equal_scenario_habitat_area,
)


all_checks_passed = (
    set(connectivity_metrics["scenario"])
    == expected_scenarios
    and connectivity_validation[
        "all_gap_distances_present"
    ].all()
    and connectivity_validation[
        "single_habitat_area_value"
    ].all()
    and connectivity_validation[
        "component_counts_positive"
    ].all()
    and connectivity_validation[
        "largest_component_percent_valid"
    ].all()
    and equal_scenario_habitat_area
)


if not all_checks_passed:
    raise ValueError(
        "One or more connectivity-analysis checks failed."
    )


print("\nAll connectivity-analysis checks passed.")

print("\nValidation table saved:")
print(VALIDATION_OUTPUT_PATH)

,scenario,number_of_gap_distances,all_gap_distances_present,single_habitat_area_value,core_habitat_area_km2,component_counts_positive,largest_component_percent_valid
0,baseline_2024,5,True,True,54.408,True,True
1,dispersed,5,True,True,66.063,True,True
2,patch_enlargement,5,True,True,66.063,True,True
3,connectivity_focused,5,True,True,66.063,True,True
4,integrated_low_matrix_pressure,5,True,True,66.063,True,True



Equal habitat area across all 2030 scenarios: True

All connectivity-analysis checks passed.

Validation table saved:
C:\Users\smit1\BlackGrouse_2030\outputs\tables\connectivity_analysis_validation.csv
